In [1]:
import glob
from joblib import Parallel, delayed
#from catalog_plotter import plot_var, get_array
import numpy as np
import matplotlib.pyplot as plt
import cartopy as cp
import cartopy.crs as ccrs
import plotter_config as pc
import sim_name_remap as map
import netCDF4 as nc4
import os
import datetime as dt
from scipy.interpolate import interp1d
import pickle as pkl
import matplotlib.tri as tri
from plotter import plot_map
import utils

In [2]:

def get_var_array(FILE, var_strs, t):

    ncfile = nc4.Dataset(FILE)
    
    if len(var_strs) > 1: 
        var_arr = []
        for sub_var in var_strs:  # Loop over all the sub variables if there are more than
            var_arr.append(ncfile.variables[sub_var][:])
        var_arr = np.nansum(np.asarray(var_arr), axis=0)  # Sum all the variables together
    else: 
        var_arr = ncfile.variables[var_strs[0]][:]

    # We assume that all these grid are unstructured.  If you are trying to plot output from a structured grid, you're going to have a bad time with this code
    if   len(np.shape(var_arr)) == 4: var_arr = np.squeeze(var_arr[t,:,:,:])
    elif len(np.shape(var_arr)) == 3: var_arr = np.squeeze(var_arr[t,:,:])
    
    lon = ncfile.variables['lon'][:]
    lat = ncfile.variables['lat'][:]
    hyam = ncfile.variables['hyam'][:]    
    hybm = ncfile.variables['hybm'][:]    
    
    return var_arr, lon, lat, hyam, hybm

def draw_box(ax, left, right, bot, top, col, thk):
    ax.plot([left, left], [bot, top], color=col, linewidth=thk)
    ax.plot([right, right], [bot, top], color=col, linewidth=thk)
    ax.plot([left, right], [bot, bot], color=col, linewidth=thk)
    ax.plot([left, right], [top, top], color=col, linewidth=thk) 


In [3]:

# This subroutine will interpolate an unstructured variable field (dimensions of lev x pt) to a specific pressure level 
# with help from the model pressure field (dimension of lev x pt, units of Pa)

def interp_to_prslev(var_arr, PS_arr, lon, lat, hyam, hybm, plot_lev, var_name):

    #print('Interpolating ' + var_name + ' to the ' + str(plot_lev) + 'hPa surface at ' + valid_datestr)
    
    var_plev = np.zeros((len(lat), len(lon)))-999
    
    for ln in range(len(lon)):
        for lt in range(len(lat)):

            curr_PS = PS_arr[lt,ln]
            curr_plev = (hyam*1.0e5 + hybm*curr_PS)/100.  

            plev_f = interp1d(curr_plev, np.squeeze(var_arr[:,lt,ln]), bounds_error=False, fill_value=-999)

            var_plev[lt,ln] = plev_f(plot_lev)
    
    return var_plev
    

In [4]:

# What are we plotting?
#sims = ['2022_ERA5_ZM_30m_COboost']  # Simulation to plot (much match a name in sim_name_remap.py)
#sims = ['2022_MERRA2_ZM_h0','2022_ERA5_ZMKE_h0','2022_MERRA2_ZMKE_h0','2022_ERA5_ZM_h0']
#sims = ['2022_M2-3hr_h0','2022_E5-1hr_h0','2022_E5-3hr_h0']
sims = ['2022_FreeRun']
plot_vars = ['CO']#,'SO2','SO4','CLDHGT']#'ISOP'] #'ZMMU','PCONVT','CLDHGT']#]  # The species we should plot (defined in plotter_config.py)     ,'SO2','SO4'
PLOTDIR = '/glade/u/home/wsmith/NudgeComparison_scripts/plots/'
start_date = '2022-08-15'
conv_dom = [110,122,36,42]   # Tiny box over the uplift region


In [5]:

for sim in sims:

    print('*****', sim)
    
    FILES = sorted(glob.glob(map.remap[sim]))   # Define all the file paths

    sim_files0 = [item for item in FILES if start_date in item][0]
    file0 = list(FILES).index(sim_files0)


    # Go file-by-file to generate plots
    for FILE in FILES[file0:]:              
    
        ncfile = nc4.Dataset(FILE)
        dates = ncfile.variables['date'][:]
        datesec = ncfile.variables['datesec'][:]
    
        # Loop over all the variables
        for plot_var_name in plot_vars:
            
            var_plot_levs = pc.var_info[plot_var_name]['plot_levs']
            var_cross_sects = pc.var_info[plot_var_name]['cross_sects']
            var_strs = pc.var_info[plot_var_name]['vars']
            var_mod = pc.var_info[plot_var_name]['mod']
                    
            # Loop over all the time steps in the current file
            for t in range(len(dates)):                                                                           ##################  TEMPORARY!!!!!!!!!!!
        
                # Define the date string at the time the plot is valid for
                valid_datestr = str(dates[t])[0:4] + '-' + str(dates[t])[4:6] + '-' + str(dates[t])[6:8] \
                + '_' + str(int(datesec[t]/3600.)).zfill(2) + 'Z_' + str(int((datesec[t]/60.) % 60)).zfill(2) + 'min'         

                print(var_plot_levs)
                
                # Get the unstructured arrays that we can use to convert to a structured grid
                var_arr, lon, lat, hyam, hybm = get_var_array(FILE, var_strs, t)
                PS_arr,  lon, lat, hyam, hybm = get_var_array(FILE, ['PS'], t)
                
                # Now loop over the map cross sections that we want to make
                for plot_lev in var_plot_levs:

                    print('Plotting ', plot_var_name, ' at ', plot_lev, ' on ', valid_datestr)
                    
                    # Interpolate the unstructured field to the desired pressure level
                    if len(np.shape(var_arr)) == 3: var_plot = interp_to_prslev(var_arr, PS_arr, lon, lat, hyam, hybm, plot_lev, plot_var_name)
                    else: var_plot = var_arr
                    
                    var_plot = var_plot*var_mod

                    fig = plt.figure(figsize=(12,7))
                    ax = fig.add_subplot(111, projection = ccrs.PlateCarree())
    
                    cf = plot_map(ax, lon, lat, var_plot, valid_datestr, plot_var_name, plot_lev, sim, PLOTDIR)

                    ax.set_title(plot_var_name + ' at ' + str(plot_lev) + 'hPa on ' + valid_datestr + ' from ' + sim, fontsize=14)

                    draw_box(ax, conv_dom[0], conv_dom[1], conv_dom[2], conv_dom[3], [0.5,0.5,0.5], 1)
                    
                    cbar_lev, cbar_rgb, norm = utils.get_colorbar(plot_var_name, plot_lev, 0)
                    cbax = fig.add_axes([0.17,0.14,0.7,0.02])
                    cbar = plt.colorbar(cf, cax=cbax, orientation='horizontal', ticks = cbar_lev[::4])
                    cbar.set_label(plot_var_name + '(' + pc.var_info[plot_var_name]['unit'] + ')', fontsize=12)
                    
                    plt.savefig(PLOTDIR + sim + '_' + plot_var_name + '_' + str(plot_lev) + 'hPa_' + valid_datestr + 'Z.png')
                    plt.close('all')
    
    
    


***** 2022_FreeRun
[200]
Plotting  CO  at  200  on  2022-08-15_03Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_06Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_09Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_12Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_15Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_18Z_00min
[200]
Plotting  CO  at  200  on  2022-08-15_21Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_00Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_03Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_06Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_09Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_12Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_15Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_18Z_00min
[200]
Plotting  CO  at  200  on  2022-08-16_21Z_00min
[200]
Plotting  CO  at  200  on  2022-08-17_00Z_00min
[200]
Plotting  CO  at  200  on  2022-08-17_03Z_00min
[200]
Plotting  CO  at  200  on  2022-08-17_06Z_00min
[200]
Plo

KeyboardInterrupt: 